# Phase 3: Verify Fine-Tuned Model

**Hardware:** T4 small  
**What this does:** Compares cosine similarity of your fine-tuned model vs the baseline
(`google/siglip-so400m-patch14-384`) on image-caption pairs from your dataset.
Fine-tuned model should score higher.

**Prerequisite:** Run `02_train_siglip.ipynb` first.

**Before running:**
1. Add secrets via the key icon (🔑) in the left sidebar — enable notebook access for each:
   - `HF_TOKEN` — HF token with read access
   - `HF_USERNAME` — your HF username
   - If your dataset is gated, accept the access request on the HF Hub page first
2. Fill in the form fields in Cell 2:
   - Click **▶ Run** on Cell 2 — a form appears **above** the code
   - Set `FINETUNED_REPO` to your fine-tuned model repo on HF Hub
   - Set `CAPTIONED_DATASET` to your captioned dataset (output of Notebook 1)
3. Run remaining cells top to bottom with **Shift+Enter**

**Pass condition:** Fine-tuned avg similarity > baseline avg similarity.  
**After passing:** Update your indexer's model name to the fine-tuned repo ID, delete cached index files, and restart the application.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets Pillow torch

In [ ]:
# @title Configuration
# @markdown Fill in your fine-tuned model repo and captioned dataset below.
FINETUNED_REPO    = "your-username/output_model" # @param {type:"string"}
CAPTIONED_DATASET = "your-username/output_ds" # @param {type:"string"}

In [ ]:
# Cell 3 — Authenticate and verify config
from google.colab import userdata
from huggingface_hub import login
import torch

HF_TOKEN    = userdata.get('HF_TOKEN')
HF_USERNAME = userdata.get('HF_USERNAME')

BASELINE = "google/siglip-so400m-patch14-384"
device   = "cuda" if torch.cuda.is_available() else "cpu"

# Authenticate — required for gated datasets
login(token=HF_TOKEN)

print("HF_TOKEN:    ", HF_TOKEN[:8] + "..." if HF_TOKEN else "ERROR: not set")
print("HF_USERNAME: ", HF_USERNAME if HF_USERNAME else "ERROR: not set")
print("Fine-tuned:  ", FINETUNED_REPO)
print("Dataset:     ", CAPTIONED_DATASET)
print("Device:      ", device)

In [ ]:
# Cell 3 — Load both models
from transformers import AutoProcessor, AutoModel

def load_model(repo):
    m = AutoModel.from_pretrained(repo, token=HF_TOKEN).to(device).eval()
    p = AutoProcessor.from_pretrained(repo, token=HF_TOKEN)
    return m, p

print("Loading fine-tuned model...")
ft_model, ft_proc = load_model(FINETUNED_REPO)
print("Loading baseline model...")
bl_model, bl_proc = load_model(BASELINE)
print("Both models loaded")

In [ ]:
# Cell 4 — Load test pairs
from datasets import load_dataset

ds = load_dataset(CAPTIONED_DATASET, split="train[:10]", token=HF_TOKEN)
print(f"Loaded {len(ds)} test pairs")
print("Sample caption:", ds[0]["caption"])

In [ ]:
# Cell 5 — Compare similarity scores
import torch.nn.functional as F

def get_similarity(model, processor, image, text):
    inp = processor(images=image, text=[text], return_tensors="pt").to(device)
    with torch.no_grad():
        img_emb = F.normalize(model.get_image_features(
            pixel_values=inp["pixel_values"]), dim=-1)
        txt_emb = F.normalize(model.get_text_features(
            input_ids=inp["input_ids"],
            attention_mask=inp["attention_mask"]), dim=-1)
    return (img_emb * txt_emb).sum().item()

ft_sims, bl_sims = [], []
for i, row in enumerate(ds):
    img     = row["image"].convert("RGB")
    caption = row["caption"]
    ft_sim  = get_similarity(ft_model, ft_proc, img, caption)
    bl_sim  = get_similarity(bl_model, bl_proc, img, caption)
    ft_sims.append(ft_sim)
    bl_sims.append(bl_sim)
    print(f"[{i+1:2d}] FT={ft_sim:.4f}  BL={bl_sim:.4f}  diff={ft_sim-bl_sim:+.4f}")

print(f"\nFine-tuned avg similarity: {sum(ft_sims)/len(ft_sims):.4f}")
print(f"Baseline  avg similarity:  {sum(bl_sims)/len(bl_sims):.4f}")
if sum(ft_sims) > sum(bl_sims):
    print("\nPASS — fine-tuned model scores higher on your domain data.")
    print(f"\nNext step: update your indexer's model name to '{FINETUNED_REPO}'")
    print("Then delete cached index files and restart the application to reindex.")
else:
    print("\nREVIEW — fine-tuned model did not outperform baseline.")
    print("Check training logs: loss should have decreased below 0.8 by epoch 3.")